# DUK leg motion workbench

This notebook is for **ground-contact mechanism motion**, not overlaid pose snapshots.

The animation uses one canonical model and updates one mechanism frame continuously, similar to a MATLAB animation loop. The core solver remains responsible for all geometry; the notebook only defines the time profile and displays results.

Model conventions:
- A is the body-frame origin: `A=(0,0)` locally; its world height changes;
- `q` is the directed CCW angle from A→B to D→A;
- B/C/O form one rigid carrier;
- the solver stays canonical; explicit world placement translates the whole leg so the tire stays tangent to z=0;
- no per-wheel adjustment, link scaling, force model or no-slip law is used.


In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

from duk_core.modeling import (
    leg_jacobian,
    load_leg_geometry_json,
    sample_leg_trajectory,
    solve_leg_pose,
)
from duk_core.modeling.ground_contact import place_leg_on_ground
from duk_core.visualization import (
    LegAnimationOptions,
    create_leg_animation,
)


def find_repo_root(start=Path.cwd()):
    """Find the repository root from any notebook working directory."""
    for candidate in (start, *start.parents):
        config = candidate / 'robot/config/physical/duk_leg_geometry.json'
        if config.exists():
            return candidate
    raise FileNotFoundError('Run this notebook from a DUK repository checkout')


repo_root = find_repo_root()
geometry = load_leg_geometry_json(
    repo_root / 'robot/config/physical/duk_leg_geometry.json'
)
geometry


## 1. Geometric workspace

The exact four-bar singular boundaries are excluded by a safety margin. This gives the interval used for the demonstration motion.


In [ ]:
lower, upper = geometry.geometric_linkage_angle_limits_rad()
margin = math.radians(5.0)
q_min = lower + margin
q_max = upper - margin

print(f'geometric q limits : {math.degrees(lower):.2f}° .. {math.degrees(upper):.2f}°')
print(f'animation q range  : {math.degrees(q_min):.2f}° .. {math.degrees(q_max):.2f}°')


## 2. Real motion animation

The signal below moves smoothly from the middle of the workspace toward both sides. `FuncAnimation` updates the same AB/CD/DA links, rigid BCO carrier, wheel and labels every frame. There is **no stack of mechanism poses**.

The embedded player provides play/pause and a timeline scrubber. `show_wheel_trace=False` keeps the default view clean.

The ground is fixed at world z=0. The wheel remains on it while A and the body move vertically. Local coordinates are kept separately for equation/invariant checks.


In [ ]:
# 5 seconds at 24 frames/s keeps the embedded player smooth and compact.
# generating an unnecessarily heavy embedded animation.
duration_s = 5.0
fps = 24
frame_count = int(duration_s * fps)
time_s = np.linspace(0.0, duration_s, frame_count, endpoint=False)

# A cosine profile starts near the middle and oscillates smoothly inside
# the safe geometric range. This is a kinematic demo profile, not a motor
# command or controller output.
q_mid = 0.5 * (q_min + q_max)
q_amplitude = 0.42 * (q_max - q_min)
q_motion = q_mid + q_amplitude * np.sin(2.0 * math.pi * time_s / duration_s)

animation = create_leg_animation(
    geometry,
    q_motion,
    options=LegAnimationOptions(
        interval_ms=round(1000 / fps),
        repeat=True,
        ground_contact=True,
        show_auxiliary=True,
        show_labels=True,
        show_wheel_trace=False,
    ),
)

# JSHTML works inside VS Code/Jupyter without requiring ffmpeg.
display(HTML(animation.to_jshtml()))
plt.close(animation.figure)


## 3. Rigid BCO verification over the motion

Animation is only presentation. The following calculation verifies that the carrier never deforms while it moves.


In [ ]:
poses = tuple(solve_leg_pose(geometry, float(q)) for q in q_motion)
carrier = geometry.bco_body


def distance(p, q):
    return math.hypot(p[0] - q[0], p[1] - q[1])


def cross2(a, b):
    return a[0] * b[1] - a[1] * b[0]


bc = np.array([distance(p.b, p.c) for p in poses])
bo = np.array([distance(p.b, p.o) for p in poses])
co = np.array([distance(p.c, p.o) for p in poses])
area2 = np.array([
    cross2(
        (p.c[0] - p.b[0], p.c[1] - p.b[1]),
        (p.o[0] - p.b[0], p.o[1] - p.b[1]),
    )
    for p in poses
])

print('max |BC error|       =', float(np.max(np.abs(bc - carrier.bc_mm))), 'mm')
print('max |BO error|       =', float(np.max(np.abs(bo - carrier.bo_mm))), 'mm')
print('max |CO error|       =', float(np.max(np.abs(co - carrier.co_mm))), 'mm')
print('max |area2 error|    =', float(np.max(np.abs(area2 - carrier.signed_double_area_mm2))), 'mm²')

world_poses = tuple(place_leg_on_ground(geometry, p) for p in poses)
ground_gaps = np.array([p.o[1] - geometry.wheel_radius_mm for p in world_poses])
assert np.max(np.abs(ground_gaps)) < 1e-9
print('max tire / ground gap =', float(np.max(np.abs(ground_gaps))), 'mm')


## 4. Motion signals

These plots explain what the animation is doing numerically: linkage coordinate, wheel-center position and wheel-center velocity.

These are **world-frame** signals. The upright wheel-center z is constant at its radius; body A moves instead. Analytical body-frame response remains in the next section.


In [ ]:
jacobians = tuple(leg_jacobian(geometry, pose) for pose in poses)
q_dot = np.gradient(q_motion, time_s)

o_x = np.array([pose.o[0] for pose in world_poses])
o_z = np.array([pose.o[1] for pose in world_poses])
body_z = np.array([pose.a[1] for pose in world_poses])
o_vx = np.array([jac.d_o[0] for jac in jacobians]) * q_dot
body_vz = -np.array([jac.d_o[1] for jac in jacobians]) * q_dot
o_vz = np.zeros_like(q_dot)

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

axes[0].plot(time_s, np.degrees(q_motion))
axes[0].set_ylabel('q (deg)')
axes[0].set_title('Kinematic motion profile')

axes[1].plot(time_s, o_x, label='Wheel O world x')
axes[1].plot(time_s, o_z, label='Wheel O world z')
axes[1].plot(time_s, body_z, label='Body A world z')
axes[1].set_ylabel('world position (mm)')
axes[1].legend()

axes[2].plot(time_s, o_vx, label='Wheel O world vx')
axes[2].plot(time_s, o_vz, label='Wheel O world vz = 0')
axes[2].plot(time_s, body_vz, label='Body A world vz')
axes[2].set_ylabel('world velocity (mm/s)')
axes[2].set_xlabel('time (s)')
axes[2].legend()

for ax in axes:
    ax.grid(True)

plt.tight_layout()
plt.show()


## 5. Workspace response

This section is analytical rather than animated. It shows how wheel-center position and transmission vary with q across the safe workspace.

The following O coordinates/Jacobians are explicitly **body-frame** quantities, not wheel height above ground.


In [ ]:
workspace_poses = sample_leg_trajectory(geometry, q_min, q_max, 181)
workspace_q_deg = np.array([
    math.degrees(pose.linkage_angle_rad)
    for pose in workspace_poses
])
workspace_jac = tuple(
    leg_jacobian(geometry, pose)
    for pose in workspace_poses
)

workspace_ox = np.array([pose.o[0] for pose in workspace_poses])
workspace_oy = np.array([pose.o[1] for pose in workspace_poses])
gain = np.array([math.hypot(jac.d_o[0], jac.d_o[1]) for jac in workspace_jac])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(workspace_q_deg, workspace_ox, label='O.x')
axes[0].plot(workspace_q_deg, workspace_oy, label='O.y')
axes[0].set_xlabel('q (deg)')
axes[0].set_ylabel('body-frame position (mm)')
axes[0].set_title('Body-frame wheel-center position')
axes[0].legend()

axes[1].plot(workspace_q_deg, gain)
axes[1].set_xlabel('q (deg)')
axes[1].set_ylabel('||dO/dq|| (mm/rad)')
axes[1].set_title('Kinematic transmission')

for ax in axes:
    ax.grid(True)

plt.tight_layout()
plt.show()


## Next motion layers

This notebook now provides true single-leg animation plus kinematic analysis. The next layers should reuse the same model for inverse kinematics, commanded height profiles, two-leg coordination and body leveling. Motor/torque/contact dynamics should be added only after their physical data and state definitions are established.
